# 🎙 Stimmenkloner – Sprachboard

Klont eine Stimme und generiert 8 Sätze als MP3-Dateien.

## ⚡ Vor dem Start
**Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern**

Dann einfach alle Zellen der Reihe nach ausführen (▶ oder Shift+Enter).

---
## Schritt 1 – Sprachprobe hochladen

- Mindestens **10–30 Sekunden** reine Sprache
- Format: MP3 oder WAV
- Kein Hintergrundgeräusch

In [ ]:
from google.colab import files

print('Datei auswählen...')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]
print(f'✓ Hochgeladen: {voice_file}')

---
## Schritt 2 – Installation
Dauert ca. 2–3 Minuten.

In [ ]:
!pip install -q f5-tts openai-whisper
!apt-get install -q ffmpeg
print('✓ Installation abgeschlossen')

---
## Schritt 3 – Alle 8 Sätze generieren

Beim ersten Mal wird das F5-TTS Modell heruntergeladen (~3 GB).

Die Sätze kannst du hier beliebig ändern.

In [ ]:
import os
import whisper
from f5_tts.api import F5TTS

saetze = [
    'Wer auf Toilette möchte, hebt bitte die Hand.',
    'Heute geht die erste Runde Bier selbstverständlich auf mich.',
    'Der Herr ist mein Hirte. Mein Fahrer ist heute Pascal.',
    'Ich erkenne ein gutes Auto daran, wie bequem der Beifahrersitz ist.',
    'Mein Lieblingsauto ist das, in dem mich andere mitnehmen.',
    'Alkoholische Mitarbeit ist heute ausdrücklich erwünscht.',
    'Ich fühle mich wie 2012 im Bierkönig.',
    'Mein Verantwortungsbereich endet ab dem zweiten Bier.',
]

# Sprachprobe auf Deutsch transkribieren (damit F5-TTS Deutsch generiert)
print('Transkribiere Sprachprobe mit Whisper...')
whisper_model = whisper.load_model('base')
result = whisper_model.transcribe(voice_file, language='de')
ref_text = result['text'].strip()
print(f'✓ Erkannt: "{ref_text[:80]}..."' if len(ref_text) > 80 else f'✓ Erkannt: "{ref_text}"')

# F5-TTS Modell laden
print('\nF5-TTS Modell wird geladen...')
tts = F5TTS()
print('✓ Modell bereit\n')

# Alle Sätze generieren
print(f'Generiere {len(saetze)} Sätze auf Deutsch...\n')
for i, text in enumerate(saetze, 1):
    wav_path = f'/content/clip_{i:02d}.wav'
    mp3_path = f'/content/clip_{i:02d}.mp3'

    tts.infer(
        ref_file=voice_file,
        ref_text=ref_text,
        gen_text=text,
        file_wave=wav_path,
        seed=42,
    )

    os.system(f'ffmpeg -i {wav_path} -q:a 2 {mp3_path} -y -loglevel quiet')
    os.remove(wav_path)

    preview = text[:60] + '...' if len(text) > 60 else text
    print(f'  ✓ clip_{i:02d}.mp3 → {preview}')

print('\n✓ Alle Clips fertig!')

---
## Schritt 4 – Herunterladen

In [ ]:
!zip -j /content/sprachboard_clips.zip /content/clip_*.mp3

from google.colab import files
files.download('/content/sprachboard_clips.zip')
print('✓ Download gestartet: sprachboard_clips.zip')

---
## Nächste Schritte
1. ZIP entpacken → `clip_01.mp3` bis `clip_08.mp3`
2. Alte Clips auf GitHub in `audio/` ersetzen
3. Fertig – Website spielt automatisch die neuen Clips